In [2]:
# Connect google drive
from google.colab import drive
ROOT = '/content/drive'
drive.mount(ROOT)

Mounted at /content/drive


# Import package

In [3]:
import os

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder )
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


# Data Loading

In [4]:
class ColabBaseConfig:
    """
    Base class to configure the root directory for Kaggle datasets.
    By default, the root_dir on Kaggle is always '/kaggle/input'.
    """
    def __init__(self, root_dir="/content/drive"):
        self.root_dir = root_dir

class ColabDataLoader(ColabBaseConfig):
    """
    Child class inheriting from KaggleBaseConfig.
    This class handles the core logic of loading a single file from a dataset.
    """
    def __init__(self, dataset_name, root_dir="/content/drive"):
        # Call the parent class constructor to set the root_dir
        super().__init__(root_dir)

        # Create the full path to the specific dataset directory
        self.dataset_name = dataset_name
        self.dataset_dir = os.path.join(self.root_dir, self.dataset_name)

    def load_csv(self, filename, **kwargs):
        """
        Read a single CSV file and return a pandas DataFrame.
        """
        file_path = os.path.join(self.dataset_dir, filename)

        # Check if the file exists before attempting to read
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found at path: {file_path}")

        df = pd.read_csv(file_path, **kwargs)

        print(f"Shape of {filename} is {df.shape}")
        return df


def load_multiple_data(data_loader, file_dict, **kwargs):
    """
    Standalone function to load multiple CSV files based on a dictionary mapping.
    """
    loaded_data = {}

    for var_name, filename in file_dict.items():
        loaded_data[var_name] = data_loader.load_csv(filename, **kwargs)

    print("All files loaded successfully!")
    return loaded_data

In [6]:
# 1. Initialize the DataLoader instance for your specific dataset
data_loader = ColabDataLoader(dataset_name="MyDrive/home-credit-default-risk")

# 2. Define your dictionary mapping variable names to their corresponding filenames
files_to_load = {
    "app_train": "application_train.csv",
    "app_test": "application_test.csv",
    "app_pre": "previous_application.csv",
    "bureau": "bureau.csv"

}

# 3. Call the standalone function, passing the loader instance and the dictionary
# This returns a dictionary with your DataFrames attached to the variable names
data = load_multiple_data(data_loader=data_loader,
                             file_dict=files_to_load)

# 4. Access your DataFrames using the keys
app_train = data["app_train"]
app_test = data["app_test"]
app_pre = data["app_pre"]

Shape of application_train.csv is (307511, 122)
Shape of application_test.csv is (48744, 121)
Shape of previous_application.csv is (1670214, 37)
Shape of bureau.csv is (1716428, 17)
All files loaded successfully!


# Preprocessing

## Utils Class/Function

In [7]:

class BaseDataProcessor:
    """
    Parent class to handle basic DataFrame inspection such as shape and memory usage.
    """
    def __init__(self, df: pd.DataFrame):
        # Store a copy of the dataframe to avoid setting with copy warning
        self.df = df.copy()

    def get_shape(self):
        """Returns the shape of the DataFrame."""
        return self.df.shape

    def get_memory_usage(self):
        """Returns the memory usage of the DataFrame in Megabytes (MB)."""
        # deep=True ensures accurate memory calculation for object types
        return self.df.memory_usage(deep=True).sum() / (1024 ** 2)


class ReduceMemProcessor(BaseDataProcessor):
    """
    Child class dedicated to reducing memory usage of a pandas DataFrame
    by downcasting numeric types and converting low-cardinality objects to categories.
    """
    def __init__(self, df: pd.DataFrame):
        super().__init__(df)

    def process(self, verbose=True):
        """
        Iterates through all columns and downcasts data types to the smallest possible safe type.
        """
        start_mem = self.get_memory_usage()

        for col in self.df.columns:
            col_type = self.df[col].dtype

            # Process numeric columns (Exclude object and existing categorical types)
            if col_type != object and not isinstance(col_type, pd.CategoricalDtype):
                c_min = self.df[col].min()
                c_max = self.df[col].max()

                # Downcast Integer types
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        self.df[col] = self.df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        self.df[col] = self.df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        self.df[col] = self.df[col].astype(np.int32)
                    elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        self.df[col] = self.df[col].astype(np.int64)

                # Downcast Float types
                else:
                    # FIX: Cast the finfo limits to native Python floats to avoid float16 overflow warnings
                    f16_min = float(np.finfo(np.float16).min)
                    f16_max = float(np.finfo(np.float16).max)
                    f32_min = float(np.finfo(np.float32).min)
                    f32_max = float(np.finfo(np.float32).max)

                    if c_min > f16_min and c_max < f16_max:
                        self.df[col] = self.df[col].astype(np.float16)
                    elif c_min > f32_min and c_max < f32_max:
                        self.df[col] = self.df[col].astype(np.float32)
                    else:
                        self.df[col] = self.df[col].astype(np.float64)

            # Process Object/String columns
            else:
                if self.df[col].nunique() < 100:
                    self.df[col] = self.df[col].astype('category')

        end_mem = self.get_memory_usage()

        if verbose:
            decrease = 100 * (start_mem - end_mem) / start_mem
            print(f"--- Memory Optimization Result ---")
            print(f"Memory usage BEFORE: {start_mem:.2f} MB")
            print(f"Memory usage AFTER : {end_mem:.2f} MB")
            print(f"Decreased by       : {decrease:.2f}%")
            print(f"----------------------------------")

        return self.df


class EncoderProcessor(BaseDataProcessor):
    """
    Child class for handling categorical encoding (e.g., One-Hot Encoding).
    """
    def __init__(self, df: pd.DataFrame):
        super().__init__(df)

    def process(self):
        """
        Applies One-Hot Encoding to all categorical columns.
        Ensures 1/0 outputs instead of True/False and cleans up unused category levels.
        """
        # 1. Clean up unused categories in existing category columns
        # This prevents ghost columns from appearing (e.g., CODE_GENDER_XNA)
        for col in self.df.columns:
            if isinstance(self.df[col].dtype, pd.CategoricalDtype):
                self.df[col] = self.df[col].cat.remove_unused_categories()

        # 2. Identify all categorical columns ('object' or 'category' types)
        categorical_cols = [
            col for col in self.df.columns
            if (self.df[col].dtype == 'object' or isinstance(self.df[col].dtype, pd.CategoricalDtype))
        ]

        # 3. Apply One-Hot Encoding
        # Explicitly passing dtype=int forces pandas to output 1/0 instead of True/False
        self.df = pd.get_dummies(
            self.df,
            columns=categorical_cols,
            drop_first=False,
            dtype=int
        )

        return self.df

class ImputerProcessor(BaseDataProcessor):
    """
    Child class for handling missing value imputation for both numeric and categorical columns.
    """
    def __init__(self, df: pd.DataFrame, numeric_strategy: str = "median", categorical_strategy: str = "mode"):
        """
        Args:
            df (pd.DataFrame): The input pandas DataFrame.
            numeric_strategy (str): Strategy for numeric columns ('median' or 'mean'). Default is 'median'.
            categorical_strategy (str): Strategy for categorical columns ('mode' or 'constant'). Default is 'mode'.
        """
        super().__init__(df)
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy

    def process(self, verbose=True):
        """
        Iterates through columns and fills missing values based on the chosen strategies.
        """
        if verbose:
            start_missing = self.df.isnull().sum().sum()
            print(f"--- Missing Value Imputation ---")
            print(f"Total missing values BEFORE: {start_missing}")

        for col in self.df.columns:
            # Skip if the column has no missing values
            if not self.df[col].isnull().any():
                continue

            col_type = self.df[col].dtype

            # 1. Handle Numeric Columns (Integers and Floats)
            if col_type != object and not isinstance(col_type, pd.CategoricalDtype):
                if self.numeric_strategy == "median":
                    fill_value = self.df[col].median()
                elif self.numeric_strategy == "mean":
                    fill_value = self.df[col].mean()
                else:
                    fill_value = 0

                self.df[col] = self.df[col].fillna(fill_value)

            # 2. Handle Categorical Columns (Object or Category type)
            else:
                if self.categorical_strategy == "mode":
                    mode_series = self.df[col].mode()
                    # Fallback to 'Missing' if the entire column is NaN and mode is empty
                    fill_value = mode_series[0] if not mode_series.empty else "Missing"
                else:
                    fill_value = "Missing"

                # Safety check for pandas 'category' dtype:
                # If the fill_value is a new string, it must be added to categories metadata first
                if isinstance(col_type, pd.CategoricalDtype) and (fill_value not in self.df[col].cat.categories):
                    self.df[col] = self.df[col].cat.add_categories(fill_value)

                self.df[col] = self.df[col].fillna(fill_value)

        if verbose:
            end_missing = self.df.isnull().sum().sum()
            print(f"Total missing values AFTER : {end_missing}")
            print(f"--------------------------------")

        return self.df

def convert_days_to_years(df: pd.DataFrame, column_trans: list) -> pd.DataFrame:
    """
    Transforms specified columns from days to years, replaces the 'DAYS_' prefix
    with 'YEARS_', and drops the old columns.

    Args:
        df (pd.DataFrame): The input pandas DataFrame.
        column_trans (list): List of column names to transform (e.g., ['DAYS_BIRTH']).

    Returns:
        pd.DataFrame: The updated DataFrame with new 'YEARS_' columns.
    """
    # Create a copy to prevent changing the original dataframe out of scope
    df = df.copy()

    for col in column_trans:
        if col in df.columns:
            # Dynamically replace only the first occurrence of 'DAYS_' with 'YEARS_'
            # This perfectly preserves single or multiple suffixes (e.g., DAYS_A_B -> YEARS_A_B)
            new_col = col.replace("DAYS_", "YEARS_", 1)

            # Convert negative days to positive years using 365 days a year
            # Cast to float32 to keep memory usage low as optimized previously
            df[new_col] = (df[col].abs() / 365.0).astype('float32')

            # Drop the original day-based column to save space
            print(f"Successfully transformed: '{col}' -> '{new_col}'")
        else:
            print(f"Warning: Column '{col}' was not found in the DataFrame.")

    return df


## App_train processing

In [20]:
app_copy = app_train.copy()


app_copy = app_copy[app_copy['CODE_GENDER'] != "XNA"] # Drop value 'XNA' in CODE_GENDER
print(f"App customer after drop 'XNA' value: {app_copy.shape}")

app_copy = app_copy[app_copy['AMT_INCOME_TOTAL'] <= 1e8] # Drop outlier in AMT_INCOME_TOTAL
print(f"App customer after drop outlier income total: {app_copy.shape}")

base_processor = BaseDataProcessor(app_copy) #Initialize and inspect the original dataframe using the base class
print(f"Original Shape: {base_processor.get_shape()}")
print(f"Original Memory: {base_processor.get_memory_usage():.2f} MB\n")


# Replace value "Nan"
if 'DAYS_EMPLOYED' in app_copy.columns:
    app_copy['DAYS_EMPLOYED'] = app_copy['DAYS_EMPLOYED'].replace(365243, np.nan)

if 'DAYS_LAST_PHONE_CHANGE' in app_copy.columns:
    app_copy['DAYS_LAST_PHONE_CHANGE'] = app_copy['DAYS_EMPLOYED'].replace(0, np.nan)

# Transform datetime day to year
days_cols = ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']
app_copy = convert_days_to_years(app_copy, days_cols)

# Ratio between YEAR_postfix and YEAR_BIRTH
app_copy['EMPLOYED_RATIO_BIRTH'] = app_copy['YEARS_EMPLOYED'] / app_copy['YEARS_BIRTH']
app_copy['REGISTRATION_RATIO_BIRTH'] = app_copy['YEARS_REGISTRATION'] / app_copy['YEARS_BIRTH']
app_copy['PUBLISH_RATIO_BIRTH'] = app_copy['YEARS_ID_PUBLISH']/ app_copy['YEARS_BIRTH']
app_copy['PHONE_CHANGE_RATIO_BIRTH'] = app_copy['YEARS_LAST_PHONE_CHANGE']/ app_copy['YEARS_BIRTH']


# Per-capita income
app_copy['INCOME_PER_PERSON'] = app_copy['AMT_INCOME_TOTAL'] / app_copy['CNT_FAM_MEMBERS'] # Total income divided by the number of family members
app_copy['CHILDREN_RATIO_FAMILY'] = app_copy['CNT_CHILDREN']/ app_copy['CNT_FAM_MEMBERS'] # Ratio between num member in family and num children

# Total contact information provided by the client (higher numbers indicate lower fraud risk)
contact_flags = ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL']
existing_contact_flags = [col for col in contact_flags if col in app_copy.columns]
if existing_contact_flags:
    app_copy['TOTAL_CONTACT_FLAGS'] = app_copy[existing_contact_flags].sum(axis=1).astype('int8')
    app_copy.drop(columns = existing_contact_flags)

# Total address mismatches (Registration vs. Living vs. Working location)
# Higher score signals lower locational stability or potential discrepancies
mismatch_flags = [
    'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION',
    'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY'
]
existing_mismatch_flags = [col for col in mismatch_flags if col in app_copy.columns]
if existing_mismatch_flags:
    app_copy['TOTAL_MISMATCH_FLAGS'] = app_copy[existing_mismatch_flags].sum(axis=1).astype('int8')
    app_copy.drop(columns = existing_mismatch_flags)

# Convert binary text flags ('Y'/'N') to numeric (1/0) if they aren't already mapped
if 'FLAG_OWN_CAR' in app_copy.columns and app_copy['FLAG_OWN_CAR'].dtype in ['object', 'category']:
    app_copy['FLAG_OWN_CAR'] = app_copy['FLAG_OWN_CAR'].map({'Y': 1, 'N': 0}).astype('int8')

if 'FLAG_OWN_REALTY' in app_copy.columns and app_copy['FLAG_OWN_REALTY'].dtype in ['object', 'category']:
    app_copy['FLAG_OWN_REALTY'] = app_copy['FLAG_OWN_REALTY'].map({'Y': 1, 'N': 0}).astype('int8')

# Combined asset score (0: No assets, 1: Owns car or house, 2: Owns both)
if 'FLAG_OWN_CAR' in app_copy.columns and 'FLAG_OWN_REALTY' in app_copy.columns:
    app_copy['TOTAL_ASSETS_SCORE'] = (app_copy['FLAG_OWN_CAR'].fillna(0) + app_copy['FLAG_OWN_REALTY'].fillna(0)).astype('int8')


app_copy['INCOME_ANNUITY_RATIO'] = (app_copy['AMT_ANNUITY'] / app_copy['AMT_INCOME_TOTAL']).astype('float32') # Measures monthly financial pressure
app_copy['CREDIT_TO_INCOME'] = (app_copy['AMT_CREDIT'] / app_copy['AMT_INCOME_TOTAL']).astype('float32') #Measures overall leverage relative to annual earnings
app_copy['CREDIT_TO_GOODS_RATIO'] = (app_copy['AMT_CREDIT'] / app_copy['AMT_GOODS_PRICE']).astype('float32') # Shows how much of the actual asset value is financed
app_copy['GOODS_CREDIT_DIFF'] = (app_copy['AMT_GOODS_PRICE'] - app_copy['AMT_CREDIT']).astype('float32') # Measures the downpayment amount (Negative means over-financing)

print("Engineering Credit Bureau aggregates...")
bureau_cols = [
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR'
]
# Safely select only the bureau columns that exist in the current dataframe
existing_bureau_cols = [col for col in bureau_cols if col in app_copy.columns]
if existing_bureau_cols:
    # Summing up all past credit checks to evaluate overall financial inquiry intensity
    app_copy['TOTAL_BUREAU_REQ'] = app_copy[existing_bureau_cols].sum(axis=1).astype('float32')

print("Engineering behavioral and timing features...")
# Flag applications submitted during weekends (Saturday and Sunday)
if 'WEEKDAY_APPR_PROCESS_START' in app_copy.columns:
    weekdays = app_copy['WEEKDAY_APPR_PROCESS_START'].astype(str).str.upper()
    app_copy['IS_WEEKEND_APPLICATION'] = weekdays.isin(['SATURDAY', 'SUNDAY']).astype('int8')

# Flag applications submitted during late hours / off-business hours (8 PM to 5 AM)
if 'HOUR_APPR_PROCESS_START' in app_copy.columns:
    app_copy['IS_LATE_REQUEST'] = app_copy['HOUR_APPR_PROCESS_START'].apply(
        lambda x: 1 if x >= 20 or x <= 5 else 0
    ).astype('int8')

# 1. Aggregate document flags into a single feature
flag_cols = [c for c in app_copy.columns if "FLAG_DOCUMENT_" in c]
app_copy["TOTAL_DOCUMENTS_SUBMITTED"] = app_copy[flag_cols].sum(axis=1) # Counts the total number of documents submitted by each applicant
app_copy.drop(columns=flag_cols, inplace=True)  # Remove redundant flag columns

# 2. Engineer External Source features
# Calculate statistical aggregations for the most critical credit score sources
ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
app_copy["EXT_SOURCE_MEAN"] = app_copy[ext_cols].mean(axis=1)
app_copy["EXT_SOURCE_PROD"] = app_copy[ext_cols].prod(axis=1)
# Keep original EXT_SOURCE columns as tree-based models highly utilize them

# 3. Aggregate Social Circle default metrics
# Capture the maximum default risk within the applicant's social circle
app_copy["MAX_DEF_SOCIAL_CIRCLE"] = app_copy[
    ["DEF_30_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE"]
].max(axis=1)
app_copy.drop(
    columns=["DEF_30_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE"],
    inplace=True,
)

# 4. Resolve Multicollinearity in Housing Data (_AVG, _MODE, _MEDI)
# Group the 3 highly correlated states of each housing attribute into 1 average feature
housing_base_features = [
    "APARTMENTS",
    "BASEMENTAREA",
    "YEARS_BEGINEXPLUATATION",
    "YEARS_BUILD",
    "COMMONAREA",
    "ELEVATORS",
    "ENTRANCES",
    "FLOORSMAX",
    "FLOORSMIN",
    "LANDAREA",
    "LIVINGAPARTMENTS",
    "LIVINGAREA",
    "NONLIVINGAPARTMENTS",
    "NONLIVINGAREA",
]

for feature in housing_base_features:
    cols_to_combine = [f"{feature}_AVG", f"{feature}_MODE", f"{feature}_MEDI"]
    # Check if the columns exist in the current DataFrame before combining
    existing_cols = [c for c in cols_to_combine if c in app_copy.columns]
    if len(existing_cols) > 0:
        # Create a combined average column to compress the data dimension
        app_copy[f"{feature}_COMBINED"] = app_copy[existing_cols].mean(axis=1)
        app_copy.drop(columns=existing_cols, inplace=True)  # Drop original states

X = app_copy.drop(columns=["SK_ID_CURR", "TARGET"], errors="ignore")
y = app_copy["TARGET"] if "TARGET" in app_copy.columns else None

# Identify column types dynamically
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
# 1. Pipeline for Numerical Features
# First, impute missing values using Median, then scale features to [0, 1] range
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# 2. Pipeline for Categorical Features
# First, impute missing text using Mode (most frequent), then convert to One-Hot vectors
cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# ColumnTransformer applies specific pipelines to the designated columns respectively
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
    ]
)

# Fit the pipelines and transform the feature matrix
X_processed = preprocessor.fit_transform(X)

# Retrieve generated column names for the final DataFrame
# This is necessary because One-Hot Encoder generates new binary column names
# encoded_cat_cols = (
#     preprocessor.named_transformers_["cat"]
#     .named_steps["onehot"]
#     .get_feature_names_out(cat_cols)
#     if cat_cols
#     else []
# )
# final_column_names = num_cols + list(encoded_cat_cols)

# # Convert the processed numpy array back to a clean Pandas DataFrame
# app_copy = pd.DataFrame(X_processed, columns=final_column_names)

# print(f"Data shape after Pipeline processing: {app_copy.shape}")

# # Reduce memory
# reducer = ReduceMemProcessor(app_copy)
# app_copy = reducer.process()

# for col in app_copy.columns:
#   if col in app_train.columns and col != 'TARGET':
#      app_copy = app_copy.drop(columns = [col])

# # # Upcast remaining float16 to float32 right after transformation
# float16_cols = app_copy.select_dtypes(include=['float16']).columns
# if len(float16_cols) > 0:
#     print(f"Upcasting {len(float16_cols)} float16 columns to float32 to fix display overflow...")
#     app_copy[float16_cols] = app_copy[float16_cols].astype('float32')
# app_copy.head()



App customer after drop 'XNA' value: (307507, 122)
App customer after drop outlier income total: (307506, 122)
Original Shape: (307506, 122)
Original Memory: 507.32 MB

Successfully transformed: 'DAYS_BIRTH' -> 'YEARS_BIRTH'
Successfully transformed: 'DAYS_EMPLOYED' -> 'YEARS_EMPLOYED'
Successfully transformed: 'DAYS_REGISTRATION' -> 'YEARS_REGISTRATION'
Successfully transformed: 'DAYS_ID_PUBLISH' -> 'YEARS_ID_PUBLISH'
Successfully transformed: 'DAYS_LAST_PHONE_CHANGE' -> 'YEARS_LAST_PHONE_CHANGE'
Engineering Credit Bureau aggregates...
Engineering behavioral and timing features...


## Bureau processing


In [22]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score  # Thay bằng metric khác nếu là bài toán khác

# 1. Chuẩn bị dữ liệu từ app_copy (Giả định bài toán Phân loại nhị phân)
# Hãy thay 'target' bằng tên cột nhãn thực tế trong tập app_copy của bạn
X = X_processed


# 2. Thiết lập tham số cấu hình cho XGBoost
# Bạn có thể điều chỉnh các tham số này thủ công thay vì dùng Grid Search
params = {
    'objective': 'binary:logistic',  # Đổi thành 'reg:squarederror' nếu là bài toán Hồi quy
    'eval_metric': 'auc',            # Đổi thành 'rmse' hoặc 'mae' nếu là bài toán Hồi quy
    'learning_rate': 0.05,           # Tốc độ học nhỏ giúp mô hình học kỹ hơn
    'max_depth': 6,                  # Độ sâu tối đa của cây
    'subsample': 0.8,                # Tỷ lệ lấy mẫu dữ liệu cho mỗi cây (tránh overfitting)
    'colsample_bytree': 0.8,         # Tỷ lệ lấy mẫu số lượng đặc trưng cho mỗi cây
    'seed': 42,
    'tree_method': 'hist',           # Sử dụng thuật toán Histogram để chạy cực nhanh
    # 'device': 'cuda'               # Mở comment dòng này nếu bạn muốn chạy bằng GPU
}

# 3. Khởi tạo K-Fold (Ví dụ chia làm 5 phần)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Khởi tạo mảng chứa dự đoán Out-of-Fold (OOF) và danh sách lưu mô hình
oof_preds = np.zeros(len(X))
models = []
fold_scores = []

print("Bắt đầu vòng lặp huấn luyện qua từng Fold...")

# 4. Vòng lặp huấn luyện chính
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n--- Đang huấn luyện Fold {fold + 1} ---")

    # Chia dữ liệu Train và Validation cho Fold hiện tại
    X_train = X.iloc[train_idx] if hasattr(X, 'iloc') else X[train_idx]
    y_train = y.iloc[train_idx] if hasattr(y, 'iloc') else y[train_idx]

    X_val = X.iloc[val_idx] if hasattr(X, 'iloc') else X[val_idx]
    y_val = y.iloc[val_idx] if hasattr(y, 'iloc') else y[val_idx]

    # Chuyển đổi dữ liệu sang định dạng DMatrix của XGBoost (tối ưu hiệu năng tối đa)
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    # Thiết lập danh sách theo dõi lỗi trên cả tập Train và Val để thực hiện Early Stopping
    watchlist = [(dtrain, 'train'), (dval, 'val')]

    # Huấn luyện mô hình cho Fold này
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=1500,          # Số vòng lặp tối đa (cứ để lớn vì đã có early stopping)
        evals=watchlist,
        early_stopping_rounds=50,      # Dừng nếu sau 50 vòng liên tiếp điểm Val không cải thiện
        verbose_eval=10             # Cứ sau 100 vòng thì in kết quả ra màn hình một lần
    )

    # Dự đoán trên tập Validation của Fold hiện tại
    val_preds = model.predict(dval)
    oof_preds[val_idx] = val_preds

    # Đánh giá hiệu năng của Fold này (Ví dụ dùng AUC)
    score = roc_auc_score(y_val, val_preds)
    fold_scores.append(score)
    models.append(model)

    print(f"-> Kết quả Fold {fold + 1} - AUC: {score:.5f}")

# 5. Tổng kết hiệu năng sau khi đi qua hết các Fold
overall_score = roc_auc_score(y, oof_preds)
print(f"\n==============================================")
print(f"Điểm AUC trung bình giữa các Fold: {np.mean(fold_scores):.5f}")
print(f"Điểm AUC Out-of-Fold (OOF) tổng thể: {overall_score:.5f}")

Bắt đầu vòng lặp huấn luyện qua từng Fold...

--- Đang huấn luyện Fold 1 ---
[0]	train-auc:0.73158	val-auc:0.72343
[100]	train-auc:0.78880	val-auc:0.75519
[200]	train-auc:0.81135	val-auc:0.75836
[300]	train-auc:0.82941	val-auc:0.75933
[400]	train-auc:0.84516	val-auc:0.75965
[404]	train-auc:0.84586	val-auc:0.75968
-> Kết quả Fold 1 - AUC: 0.75968

--- Đang huấn luyện Fold 2 ---
[0]	train-auc:0.73220	val-auc:0.72306
[100]	train-auc:0.78890	val-auc:0.75503
[200]	train-auc:0.81090	val-auc:0.75898
[300]	train-auc:0.82861	val-auc:0.76004
[400]	train-auc:0.84397	val-auc:0.76038
[422]	train-auc:0.84700	val-auc:0.76015
-> Kết quả Fold 2 - AUC: 0.76015

--- Đang huấn luyện Fold 3 ---
[0]	train-auc:0.73183	val-auc:0.72118
[100]	train-auc:0.78853	val-auc:0.75669
[200]	train-auc:0.81124	val-auc:0.76145
[300]	train-auc:0.82868	val-auc:0.76260
[400]	train-auc:0.84491	val-auc:0.76326
[470]	train-auc:0.85491	val-auc:0.76348
-> Kết quả Fold 3 - AUC: 0.76348

--- Đang huấn luyện Fold 4 ---
[0]	train-auc:

#